In [1]:
import torch
import torch.nn as nn
import torch.optim as optim 
import torchvision
from torchvision.datasets import CIFAR10

from torch.utils.data import DataLoader
import torchvision.transforms as transforms

# image => scale (0, 1) => normalize => (-1, 1)
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

trainset = CIFAR10(root= "./data", train= True, download= True, transform= transform)
testset = CIFAR10(root= "./data", train= False, download= True, transform= transform)

In [2]:
trainloader = DataLoader(trainset, batch_size= 64, shuffle= True)
testloader = DataLoader(testset, batch_size= 63)

In [3]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()

        self.conv_layers = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size= 3, padding= 1), #(input channel = 3 , output channel 32) 
            nn.ReLU(),
            nn.MaxPool2d(2, 2), # kernal , stride

            nn.Conv2d(32, 64, kernel_size= 3, padding= 1), #(double the output channel)
            nn.ReLU(),
            nn.MaxPool2d(2, 2), # kernal , stride

            nn.Conv2d(64, 128, kernel_size= 3, padding= 1), #(double output channel from previous one)
            nn.ReLU(),
            nn.MaxPool2d(2, 2), # kernal , stride

            
        )

        self.fc_layers = nn.Sequential(
            nn.Linear(4*4*128, 256), # maxpool converts the height and width into half, so at third layer it becomes 4,4(height and width) and 256 is ht output we want.
            nn.ReLU(),
            nn.Linear(256, 10),
        )

    def forward(self, x):
        x = self.conv_layers(x)
        x = x.view(x.size(0), -1) # flattening
        x = self.fc_layers(x)
        return x


In [4]:
model = CNN()

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

In [5]:
epochs = 10

for epoch in range(epochs):
    epoch_training_loss = 0.0

    for images, labels in trainloader:
        optimizer.zero_grad()

        output = model.forward(images)
        loss = criterion(output, labels)
        loss.backward()
        optimizer.step()

        epoch_training_loss += loss.item()

    print(f" epoch = {epoch+1}/{epochs} & loss= {epoch_training_loss/len(trainloader)}")

 epoch = 1/10 & loss= 1.3614152434384426
 epoch = 2/10 & loss= 0.9202137719029966
 epoch = 3/10 & loss= 0.7371661021657612
 epoch = 4/10 & loss= 0.6066100348139662
 epoch = 5/10 & loss= 0.5055677281201952
 epoch = 6/10 & loss= 0.40595083452208575
 epoch = 7/10 & loss= 0.3163281261463604
 epoch = 8/10 & loss= 0.24642007757464182
 epoch = 9/10 & loss= 0.18195234907462315
 epoch = 10/10 & loss= 0.14582470925691562


In [14]:
# Evaludate our CNN

correct_labels = 0
total_labels = 0

model.eval()

with torch.no_grad():
    for images, labels in testloader:
        outputs = model.forward(images)
        _, predicted  = torch.max(outputs, 1)

        correct_labels += (predicted == labels).sum().item()
        total_labels += labels.size(0)

print(f"accuracy = {correct_labels / total_labels * 100}")

accuracy = 75.69
